# MedBIND3D v16b — Larger Model (base=32) + 100 Epochs

v16 with base=16 underperformed (clean ET=0.68 vs nnU-Net 0.82) due to model being too small and undertrained.
This notebook fixes both: base=32 (4M params, still fits 8GB VRAM) trained for 100 epochs.

**Place this notebook in the same directory as v16:**
`C:/Users/arnav/Desktop/MedBIND3D/MedBIND3D/medclipsam/MedCLIP-SAMv2/MedBIND3D_v16b.ipynb`

In [1]:
# ── CELL 1: Setup ──────────────────────────────────────────────────────────────
import numpy as np, pandas as pd, torch, torch.nn as nn
import torch.nn.functional as F, nibabel as nib, cv2
from pathlib import Path
from tqdm import tqdm
from scipy.stats import ttest_rel
import os, warnings, shutil, random, traceback, gc
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

os.chdir('C:/Users/arnav/Desktop/MedBIND3D/MedBIND3D/medclipsam/MedCLIP-SAMv2')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_ROOT  = 'C:/Users/arnav/Desktop/MedBIND3D/MedBIND3D/BraTS2020_training_data/MICCAI_BraTS2020_TrainingData'
OUTPUT_DIR = './medbind3d_v16_outputs'
MODEL_CKPT = f'{OUTPUT_DIR}/v16b_final.pth'

PATCH_H, PATCH_W, PATCH_D = 96, 96, 96
MODS    = ['FLAIR','T1','T1CE','T2']
REGIONS = ['WT','TC','ET']
N_TEST  = 20

Path(OUTPUT_DIR).mkdir(exist_ok=True)
print(f'Device: {device}')
print(f'Outputs -> {os.path.abspath(OUTPUT_DIR)}')

Device: cuda
Outputs -> C:\Users\arnav\Desktop\MedBIND3D\MedBIND3D\medclipsam\MedCLIP-SAMv2\medbind3d_v16_outputs


In [2]:
# ── CELL 2: Dataset (memory-efficient — one modality at a time) ───────────────
def get_all_patient_dirs(root):
    return [d for d in sorted(Path(root).iterdir())
            if d.is_dir() and all(len(list(d.glob(f'*{s}*')))>0
            for s in ['t1.nii','t1ce.nii','t2.nii','flair.nii','seg.nii'])]

def load_seg_only(patient_dir):
    seg_f = list(patient_dir.glob('*seg.nii*'))[0]
    return nib.load(str(seg_f)).get_fdata(dtype=np.float32).astype(np.int16)

def load_patient_patch(patient_dir, h0, w0, d0,
                        ph=PATCH_H, pw=PATCH_W, pd_sz=PATCH_D):
    """Load ONLY the patch region for each modality — never loads full volume."""
    vols = []
    for m in ['flair','t1','t1ce','t2']:
        f = list(patient_dir.glob(f'*{m}.nii*'))[0]
        img = nib.load(str(f))
        # Use slicer to load only the patch region — avoids loading full volume
        slicer = (slice(h0, h0+ph), slice(w0, w0+pw), slice(d0, d0+pd_sz))
        v = np.asanyarray(img.dataobj)[slicer].astype(np.float32)
        if v.shape != (ph, pw, pd_sz):
            pad = [(0, ph-v.shape[0]), (0, pw-v.shape[1]), (0, pd_sz-v.shape[2])]
            v = np.pad(v, pad)
        mask = v > 0
        if mask.sum() > 0:
            v[mask] = (v[mask]-v[mask].mean())/(v[mask].std()+1e-8)
        vols.append(v)
        del img
    return np.stack(vols, axis=0)  # [4, ph, pw, pd_sz] — tiny, not full volume

def get_volume_shape(patient_dir):
    f = list(patient_dir.glob('*flair.nii*'))[0]
    return nib.load(str(f)).shape  # just reads header, no data

def get_random_patch_coords(patient_dir, ph=PATCH_H, pw=PATCH_W, pd_sz=PATCH_D, tumor_bias=0.7):
    """Get patch coordinates using tumor-centered bias, loading seg minimally."""
    H, W, D = get_volume_shape(patient_dir)
    if random.random() < tumor_bias:
        seg = load_seg_only(patient_dir)
        tumor_voxels = np.argwhere(seg > 0)
        del seg; gc.collect()
        if len(tumor_voxels) > 0:
            center = tumor_voxels[np.random.randint(len(tumor_voxels))]
            h0 = int(np.clip(center[0]-ph//2, 0, max(0, H-ph)))
            w0 = int(np.clip(center[1]-pw//2, 0, max(0, W-pw)))
            d0 = int(np.clip(center[2]-pd_sz//2, 0, max(0, D-pd_sz)))
            return h0, w0, d0
    h0=random.randint(0,max(0,H-ph))
    w0=random.randint(0,max(0,W-pw))
    d0=random.randint(0,max(0,D-pd_sz))
    return h0, w0, d0

def load_seg_patch(patient_dir, h0, w0, d0, ph=PATCH_H, pw=PATCH_W, pd_sz=PATCH_D):
    seg_f = list(patient_dir.glob('*seg.nii*'))[0]
    img = nib.load(str(seg_f))
    slicer = (slice(h0, h0+ph), slice(w0, w0+pw), slice(d0, d0+pd_sz))
    seg = np.asanyarray(img.dataobj)[slicer].astype(np.int16)
    if seg.shape != (ph, pw, pd_sz):
        seg = np.pad(seg, [(0,ph-seg.shape[0]),(0,pw-seg.shape[1]),(0,pd_sz-seg.shape[2])])
    return seg

all_dirs   = get_all_patient_dirs(DATA_ROOT)
test_dirs  = all_dirs[:N_TEST]
train_dirs = all_dirs[N_TEST:]
print(f'Train: {len(train_dirs)}  Test: {len(test_dirs)}')
print('Memory-efficient loading: patches only, never full volumes during training')

Train: 348  Test: 20
Memory-efficient loading: patches only, never full volumes during training


In [3]:
# ── CELL 3: Architecture (base=32, 4M params) ─────────────────────────────────
class ConvBlock3D(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(cin, cout, 3, padding=1, bias=False),
            nn.InstanceNorm3d(cout), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(cout, cout, 3, padding=1, bias=False),
            nn.InstanceNorm3d(cout), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.net(x)

class UNet3DModalityDropout(nn.Module):
    def __init__(self, in_ch=4, num_classes=4, base=32):
        super().__init__()
        self.enc1 = ConvBlock3D(in_ch,   base)
        self.enc2 = ConvBlock3D(base,    base*2)
        self.enc3 = ConvBlock3D(base*2,  base*4)
        self.enc4 = ConvBlock3D(base*4,  base*8)
        self.pool = nn.MaxPool3d(2)
        self.up3  = nn.ConvTranspose3d(base*8, base*4, 2, stride=2)
        self.dec3 = ConvBlock3D(base*8,  base*4)
        self.up2  = nn.ConvTranspose3d(base*4, base*2, 2, stride=2)
        self.dec2 = ConvBlock3D(base*4,  base*2)
        self.up1  = nn.ConvTranspose3d(base*2, base,   2, stride=2)
        self.dec1 = ConvBlock3D(base*2,  base)
        self.out  = nn.Conv3d(base, num_classes, 1)
    def forward(self, x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1))
        e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3))
        d3=self.dec3(torch.cat([self.up3(e4),e3],1))
        d2=self.dec2(torch.cat([self.up2(d3),e2],1))
        d1=self.dec1(torch.cat([self.up1(d2),e1],1))
        return self.out(d1)

model = UNet3DModalityDropout(in_ch=4, num_classes=4, base=32).to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

# VRAM check
_x = torch.randn(1, 4, PATCH_H, PATCH_W, PATCH_D).to(device)
_o = model(_x)
assert _o.shape == (1, 4, PATCH_H, PATCH_W, PATCH_D)
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated()/1e9
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'VRAM: {used:.2f}GB / {total:.2f}GB')
    assert used < total*0.85, f'VRAM too close to limit ({used:.2f}/{total:.2f}GB). Reduce batch size.'
del _x, _o; gc.collect(); torch.cuda.empty_cache()
print('SMOKE TEST PASSED: architecture fits in VRAM')

Model params: 5.60M
VRAM: 1.60GB / 8.59GB
SMOKE TEST PASSED: architecture fits in VRAM


In [4]:
# ── CELL 4: Training Functions ─────────────────────────────────────────────────
def apply_modality_dropout(x, dropout_prob=0.5):
    mask = (torch.rand(x.shape[0], x.shape[1], 1, 1, 1, device=x.device) > dropout_prob).float()
    for b in range(x.shape[0]):
        if mask[b].sum() < 2:
            restore = torch.randperm(4)[:2]
            mask[b, restore] = 1.0
    return x * mask

def random_3d_crop(vol4ch, seg, ph=PATCH_H, pw=PATCH_W, pd_sz=PATCH_D):
    H, W, D = seg.shape
    if seg.sum() > 0 and random.random() < 0.7:
        tumor_voxels = np.argwhere(seg > 0)
        center = tumor_voxels[np.random.randint(len(tumor_voxels))]
        h0 = int(np.clip(center[0]-ph//2, 0, max(0,H-ph)))
        w0 = int(np.clip(center[1]-pw//2, 0, max(0,W-pw)))
        d0 = int(np.clip(center[2]-pd_sz//2, 0, max(0,D-pd_sz)))
    else:
        h0=random.randint(0,max(0,H-ph))
        w0=random.randint(0,max(0,W-pw))
        d0=random.randint(0,max(0,D-pd_sz))
    pv = vol4ch[:, h0:h0+ph, w0:w0+pw, d0:d0+pd_sz]
    ps = seg[h0:h0+ph, w0:w0+pw, d0:d0+pd_sz]
    if ps.shape != (ph,pw,pd_sz):
        pad_v=[(0,0),(0,ph-pv.shape[1]),(0,pw-pv.shape[2]),(0,pd_sz-pv.shape[3])]
        pv=np.pad(pv,pad_v); ps=np.pad(ps,pad_v[1:])
    return pv, ps

def seg_to_onehot(seg_np, n_classes=4):
    s=seg_np.copy(); s[s==4]=3
    onehot=np.zeros((n_classes,)+s.shape, dtype=np.float32)
    for c in range(n_classes): onehot[c]=(s==c)
    return onehot

def soft_dice_loss_multiclass(logits, target_onehot, weights=[0.5,1.0,1.0,2.0]):
    probs=torch.softmax(logits,dim=1); loss=0.0
    for c,w in enumerate(weights):
        p=probs[:,c].reshape(-1); t=target_onehot[:,c].reshape(-1)
        i=(p*t).sum(); u=p.sum()+t.sum()
        loss += w*(1-(2*i+1)/(u+1))
    return loss/sum(weights)

def random_flip_3d(vol, seg):
    axes=[ax for ax in [2,3,4] if random.random()>0.5]
    for ax in axes:
        vol=torch.flip(vol,[ax]); seg=torch.flip(seg,[ax-1])
    return vol, seg

print('Training functions ready')

Training functions ready


In [7]:
# ── Pre-cache tumor centers (run ONCE before Cell 5) ─────────────────────────
# Stores only voxel coordinates, not volumes — ~1KB per patient, negligible RAM

print('Pre-caching tumor center coordinates...')
tumor_centers_cache = {}   # pid -> list of (h,w,d) tumor voxel coords (subsampled)
vol_shape_cache = {}       # pid -> (H,W,D)

for pd_ in tqdm(train_dirs + test_dirs, desc='Caching coords'):
    pid = pd_.name
    try:
        H, W, D = get_volume_shape(pd_)
        vol_shape_cache[pid] = (H, W, D)
        seg = load_seg_only(pd_)
        coords = np.argwhere(seg > 0)
        # Subsample to max 500 coords to keep RAM tiny
        if len(coords) > 500:
            idx = np.random.choice(len(coords), 500, replace=False)
            coords = coords[idx]
        tumor_centers_cache[pid] = coords
        del seg
    except Exception as e:
        vol_shape_cache[pid] = (240, 240, 155)
        tumor_centers_cache[pid] = np.array([])
        print(f'  [FAILED] {pid}: {e}')

gc.collect()
print(f'Done: {len(tumor_centers_cache)} patients cached')

Pre-caching tumor center coordinates...


Caching coords: 100%|██████████| 368/368 [00:31<00:00, 11.77it/s]


Done: 368 patients cached


In [8]:
# ── Updated get_random_patch_coords (reads from cache, no disk I/O) ───────────
def get_random_patch_coords(patient_dir, ph=PATCH_H, pw=PATCH_W,
                             pd_sz=PATCH_D, tumor_bias=0.7):
    pid = patient_dir.name
    H, W, D = vol_shape_cache.get(pid, (240, 240, 155))
    coords = tumor_centers_cache.get(pid, np.array([]))
    if len(coords) > 0 and random.random() < tumor_bias:
        center = coords[np.random.randint(len(coords))]
        h0 = int(np.clip(center[0]-ph//2, 0, max(0, H-ph)))
        w0 = int(np.clip(center[1]-pw//2, 0, max(0, W-pw)))
        d0 = int(np.clip(center[2]-pd_sz//2, 0, max(0, D-pd_sz)))
        return h0, w0, d0
    return (random.randint(0, max(0, H-ph)),
            random.randint(0, max(0, W-pw)),
            random.randint(0, max(0, D-pd_sz)))

print('get_random_patch_coords updated — reads from cache, zero disk I/O')

get_random_patch_coords updated — reads from cache, zero disk I/O


In [9]:
# ── CELL 5: TRAIN (100 epochs, base=32) ───────────────────────────────────────
N_EPOCHS = 100
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, N_EPOCHS)

# Resume from checkpoint if exists
start_epoch = 0
for ep in [90,80,70,60,50,40,30,20,10]:
    ckpt = f'{OUTPUT_DIR}/v16b_epoch{ep}.pth'
    if Path(ckpt).exists():
        model.load_state_dict(torch.load(ckpt, map_location=device))
        start_epoch = ep
        # Fast-forward scheduler
        for _ in range(ep): scheduler.step()
        print(f'Resumed from epoch {ep} checkpoint')
        break
if start_epoch == 0:
    print('Training from scratch')

# Smoke test
print('Running smoke test...')
_h0,_w0,_d0 = get_random_patch_coords(train_dirs[0])
_pv = load_patient_patch(train_dirs[0], _h0, _w0, _d0)
_ps = load_seg_patch(train_dirs[0], _h0, _w0, _d0)
_x = apply_modality_dropout(torch.FloatTensor(_pv).unsqueeze(0).to(device))
_y = torch.FloatTensor(seg_to_onehot(_ps)).unsqueeze(0).to(device)
_l = soft_dice_loss_multiclass(model(_x),_y)
_l.backward(); optimizer.zero_grad()
print(f'SMOKE TEST PASSED: loss={_l.item():.4f}')
del _pv,_ps,_x,_y,_l
gc.collect(); torch.cuda.empty_cache()

print(f'\nTraining epochs {start_epoch+1}-{N_EPOCHS} on {len(train_dirs)} patients')
print(f'Expected time: ~{(N_EPOCHS-start_epoch)*len(train_dirs)*4*2.5/3600:.1f} hours\n')

train_losses = []
for epoch in range(start_epoch, N_EPOCHS):
    model.train()
    ep_loss=0.0; n_steps=0; n_failed=0
    random.shuffle(train_dirs)
    for pd_ in tqdm(train_dirs, desc=f'Epoch {epoch+1}/{N_EPOCHS}', leave=False):
            try:
                for _ in range(4):
                    # Get patch coords (loads seg briefly for tumor bias, then discards)
                    h0, w0, d0 = get_random_patch_coords(pd_)
                    # Load only the patch — peak RAM: ~34MB not 136MB
                    patch_v = load_patient_patch(pd_, h0, w0, d0)
                    patch_s = load_seg_patch(pd_, h0, w0, d0)

                    x = torch.FloatTensor(patch_v).unsqueeze(0).to(device)
                    y = torch.FloatTensor(seg_to_onehot(patch_s)).unsqueeze(0).to(device)
                    if random.random() < 0.5:
                        x = apply_modality_dropout(x, dropout_prob=0.5)
                    x, _ = random_flip_3d(x, torch.FloatTensor(patch_s).unsqueeze(0))
                    optimizer.zero_grad()
                    logits = model(x)
                    loss = soft_dice_loss_multiclass(logits, y)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    ep_loss += loss.item(); n_steps += 1
                    del patch_v, patch_s, x, y, logits
            except Exception as e:
                n_failed += 1; print(f'  [FAILED] {pd_.name}: {e}'); continue
    gc.collect(); torch.cuda.empty_cache()
    if n_steps==0: raise RuntimeError(f'Epoch {epoch+1}: zero steps. STOPPING.')
    scheduler.step()
    avg=ep_loss/n_steps; train_losses.append(avg)
    print(f'Epoch {epoch+1:3d}/{N_EPOCHS}  Loss: {avg:.4f}  LR: {scheduler.get_last_lr()[0]:.6f}  (failed={n_failed})')
    if (epoch+1)%10==0:
        torch.save(model.state_dict(), f'{OUTPUT_DIR}/v16b_epoch{epoch+1}.pth')
        print(f'  Checkpoint saved: epoch {epoch+1}')

torch.save(model.state_dict(), MODEL_CKPT)
plt.figure(figsize=(7,3))
plt.plot(train_losses); plt.grid(alpha=0.3)
plt.xlabel('Epoch'); plt.ylabel('Dice loss')
plt.title('v16b: base=32, 100 epochs, modality dropout')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v16b_training_curve.png', dpi=300, bbox_inches='tight')
plt.close()
print(f'\nModel saved -> {MODEL_CKPT}')

Training from scratch
Running smoke test...
SMOKE TEST PASSED: loss=0.9191

Training epochs 1-100 on 348 patients
Expected time: ~96.7 hours



Epoch   1/100  Loss: 0.8185  LR: 0.001000  (failed=0)


Epoch   2/100  Loss: 0.7963  LR: 0.000999  (failed=0)


Epoch   3/100  Loss: 0.7947  LR: 0.000998  (failed=0)


Epoch   4/100  Loss: 0.7832  LR: 0.000996  (failed=0)


...

Epoch  96/100  Loss: 0.6510  LR: 0.000004  (failed=0)


Epoch  97/100  Loss: 0.6448  LR: 0.000002  (failed=0)


Epoch  98/100  Loss: 0.6465  LR: 0.000001  (failed=0)


Epoch  99/100  Loss: 0.6530  LR: 0.000000  (failed=0)


Epoch 100/100  Loss: 0.6441  LR: 0.000000  (failed=0)
  Checkpoint saved: epoch 100

Model saved -> ./medbind3d_v16_outputs/v16b_final.pth


In [11]:
# ── CELL 6: Sliding Window Inference + Load Model ─────────────────────────────
model.load_state_dict(torch.load(MODEL_CKPT, map_location=device))
model.eval()
print('Model loaded')

def load_full_volume(patient_dir):
    """Load full volume for inference (test time — RAM freed from training)."""
    vols = np.stack([
        nib.load(str(list(patient_dir.glob(f'*{m}.nii*'))[0])).get_fdata(dtype=np.float32)
        for m in ['flair','t1','t1ce','t2']], axis=0)
    for i in range(4):
        mask = vols[i] > 0
        if mask.sum() > 0:
            vols[i][mask] = (vols[i][mask]-vols[i][mask].mean())/(vols[i][mask].std()+1e-8)
    seg = nib.load(str(list(patient_dir.glob('*seg.nii*'))[0])).get_fdata().astype(np.int16)
    return vols, seg

def gt_regions(seg):
    return {'WT':(seg>0).astype(np.float32),
            'TC':((seg==1)|(seg==4)).astype(np.float32),
            'ET':(seg==4).astype(np.float32)}

def dice_3d(pred, gt):
    i=np.sum(pred*gt); u=np.sum(pred)+np.sum(gt)
    return 2.0*i/u if u>0 else 0.0

def sliding_window_inference(model, vol4ch, ph=PATCH_H, pw=PATCH_W,
                              pd_sz=PATCH_D, stride=48):
    H,W,D = vol4ch.shape[1],vol4ch.shape[2],vol4ch.shape[3]
    probs_sum = np.zeros((4,H,W,D), dtype=np.float32)
    count     = np.zeros((1,H,W,D), dtype=np.float32)
    h_steps = sorted(set(list(range(0,max(1,H-ph+1),stride))+[max(0,H-ph)]))
    w_steps = sorted(set(list(range(0,max(1,W-pw+1),stride))+[max(0,W-pw)]))
    d_steps = sorted(set(list(range(0,max(1,D-pd_sz+1),stride))+[max(0,D-pd_sz)]))
    model.eval()
    with torch.no_grad():
        for h0 in h_steps:
            for w0 in w_steps:
                for d0 in d_steps:
                    h1=min(h0+ph,H); w1=min(w0+pw,W); d1=min(d0+pd_sz,D)
                    patch=vol4ch[:,h0:h1,w0:w1,d0:d1]
                    rh,rw,rd=h1-h0,w1-w0,d1-d0
                    if rh<ph or rw<pw or rd<pd_sz:
                        patch=np.pad(patch,[(0,0),(0,ph-rh),(0,pw-rw),(0,pd_sz-rd)])
                    x=torch.FloatTensor(patch).unsqueeze(0).to(device)
                    logits=model(x)
                    probs=torch.softmax(logits,dim=1)[0].cpu().numpy()
                    probs_sum[:,h0:h1,w0:w1,d0:d1]+=probs[:,:rh,:rw,:rd]
                    count[0,h0:h1,w0:w1,d0:d1]+=1
    p=probs_sum/(count+1e-8)
    return {'WT':((p[1]+p[2]+p[3])>0.5).astype(float),
            'TC':((p[1]+p[3])>0.5).astype(float),
            'ET':(p[3]>0.5).astype(float)}

# Smoke test
print('Running inference smoke test...')
_vols,_seg = load_full_volume(test_dirs[0])
_gtr = gt_regions(_seg)
_preds = sliding_window_inference(model, _vols)
for r in REGIONS:
    print(f'  {r}: {dice_3d(_preds[r],_gtr[r]):.3f}')
del _vols,_seg,_gtr,_preds
gc.collect(); torch.cuda.empty_cache()
print('Smoke test passed')

Model loaded
Running inference smoke test...
  WT: 0.792
  TC: 0.867
  ET: 0.843
Smoke test passed


In [12]:
# ── CELL 7: EVALUATION — All 5 Scenarios ──────────────────────────────────────
def corrupt_vol(vols, ch_idx, sigma=0.5):
    out=vols.copy()
    noise=np.random.randn(*out[ch_idx].shape).astype(np.float32)*sigma
    out[ch_idx]=(out[ch_idx]+noise)
    return out

scenarios = {
    'clean':         lambda v: v.copy(),
    'missing_T1CE':  lambda v: np.concatenate([v[:2],np.zeros_like(v[2:3]),v[3:]],axis=0),
    'missing_FLAIR': lambda v: np.concatenate([np.zeros_like(v[:1]),v[1:]],axis=0),
    'corrupt_T1CE':  lambda v: corrupt_vol(v,2),
    'corrupt_FLAIR': lambda v: corrupt_vol(v,0),
}

eval_rows=[]
print('='*80)
print('MedBIND3D v16b — 3D U-Net base=32, 100 epochs')
print('='*80)

for pd_ in tqdm(test_dirs, desc='Test patients'):
    pid=pd_.name
    try:
        vols, seg = load_full_volume(pd_)
        gtr = gt_regions(seg)
        row = {'Patient':pid}
        for scenario_name,transform_fn in scenarios.items():
            vols_s = transform_fn(vols)
            preds  = sliding_window_inference(model, vols_s)
            for r in REGIONS:
                row[f'{r}_{scenario_name}'] = dice_3d(preds[r],gtr[r])
            gc.collect(); torch.cuda.empty_cache()
        eval_rows.append(row)
        pd.DataFrame(eval_rows).to_csv(f'{OUTPUT_DIR}/v16b_results.csv',index=False)
        print(f'  {pid}: clean ET={row["ET_clean"]:.3f}  miss_T1CE ET={row["ET_missing_T1CE"]:.3f}')
    except Exception as e:
        print(f'  [FAILED] {pid}: {e}'); traceback.print_exc()
    finally:
        try: del vols,seg,gtr
        except: pass
        gc.collect()

df=pd.DataFrame(eval_rows)
print('\n'+'='*80)
for scenario_name in scenarios.keys():
    print(f'\n--- {scenario_name.upper()} ---')
    for r in REGIONS:
        col=f'{r}_{scenario_name}'; clean_col=f'{r}_clean'
        if col not in df.columns: continue
        m=df[col].mean(); s=df[col].std()
        drop=df[clean_col].mean()-m if scenario_name!='clean' else 0
        print(f'  {r}: {m:.4f}+/-{s:.4f}  (drop: {drop:+.4f})')

MedBIND3D v16b — 3D U-Net base=32, 100 epochs


Test patients:   5%|▌         | 1/20 [00:42<13:25, 42.40s/it]

  BraTS20_Training_001: clean ET=0.843  miss_T1CE ET=0.125
  BraTS20_Training_002: clean ET=0.600  miss_T1CE ET=0.188


Test patients:  10%|█         | 2/20 [01:29<13:35, 45.31s/it]

  BraTS20_Training_003: clean ET=0.645  miss_T1CE ET=0.033


...

  BraTS20_Training_017: clean ET=0.709  miss_T1CE ET=0.173
  BraTS20_Training_018: clean ET=0.577  miss_T1CE ET=0.152


Test patients:  95%|█████████▌| 19/20 [14:38<00:46, 46.54s/it]

  BraTS20_Training_019: clean ET=0.697  miss_T1CE ET=0.442


Test patients: 100%|██████████| 20/20 [15:25<00:00, 46.27s/it]

  BraTS20_Training_020: clean ET=0.577  miss_T1CE ET=0.331


--- CLEAN ---
  WT: 0.7162+/-0.1165  (drop: +0.0000)
  TC: 0.7699+/-0.1457  (drop: +0.0000)
  ET: 0.6987+/-0.1227  (drop: +0.0000)

--- MISSING_T1CE ---
  WT: 0.6610+/-0.1237  (drop: +0.0552)
  TC: 0.4333+/-0.1806  (drop: +0.3366)
  ET: 0.2703+/-0.1575  (drop: +0.4284)

--- MISSING_FLAIR ---
  WT: 0.5832+/-0.1432  (drop: +0.1330)
  TC: 0.7183+/-0.1781  (drop: +0.0516)
  ET: 0.5963+/-0.2710  (drop: +0.1023)

--- CORRUPT_T1CE ---
  WT: 0.7291+/-0.1122  (drop: -0.0129)
  TC: 0.7723+/-0.1472  (drop: -0.0024)
  ET: 0.6974+/-0.1123  (drop: +0.0013)

--- CORRUPT_FLAIR ---
  WT: 0.7174+/-0.1147  (drop: -0.0012)
  TC: 0.7690+/-0.1460  (drop: +0.0009)
  ET: 0.6979+/-0.1219  (drop: +0.0008)


In [13]:
# ── CELL 8: COMPARISON vs nnU-Net + SOTA ──────────────────────────────────────
df = pd.read_csv(f'{OUTPUT_DIR}/v16b_results.csv')
matplotlib.rc('font',family='serif',size=9)
CR=['#3A7D44','#E8871E','#8B5E83']
scenario_labels=['Clean','Miss T1CE','Miss FLAIR','Corr T1CE','Corr FLAIR']
scenario_cols  =['clean','missing_T1CE','missing_FLAIR','corrupt_T1CE','corrupt_FLAIR']

fig,axes=plt.subplots(1,3,figsize=(12,3.5))
for ax,r,c in zip(axes,REGIONS,CR):
    means=[df[f'{r}_{s}'].mean() for s in scenario_cols]
    stds =[df[f'{r}_{s}'].std() for s in scenario_cols]
    ax.bar(scenario_labels,means,yerr=stds,capsize=3,color=c,alpha=0.85,edgecolor='black',linewidth=0.5)
    ax.axhline(means[0],color='gray',linestyle='--',linewidth=0.8,alpha=0.5)
    ax.set_title(f'{r}'); ax.set_ylabel('Dice'); ax.set_ylim(0,1.0)
    ax.grid(axis='y',alpha=0.3); ax.spines[['top','right']].set_visible(False)
    for lbl in ax.get_xticklabels(): lbl.set_rotation(20); lbl.set_ha('right')
plt.suptitle('v16b: 3D U-Net with modality dropout (base=32, 100 epochs)',y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v16b_fig_results.png',dpi=400,bbox_inches='tight')
plt.close()

print('='*80)
print('COMPARISON TABLE')
print('='*80)
print(f'{"Method":<35}{"WT":<10}{"TC":<10}ET')
print('-'*65)
# nnU-Net baselines from memory
nnunet = {'clean':(0.890,0.901,0.819),'missing_T1CE':(0.888,0.797,0.595)}
for k,(wt,tc,et) in nnunet.items():
    print(f'{"nnU-Net+TTA ("+k+")":<35}{wt:<10.3f}{tc:<10.3f}{et:.3f}')
print()
for scenario in ['clean','missing_T1CE','missing_FLAIR']:
    wt=df[f'WT_{scenario}'].mean(); tc=df[f'TC_{scenario}'].mean(); et=df[f'ET_{scenario}'].mean()
    print(f'{"v16b ("+scenario+")":<35}{wt:<10.3f}{tc:<10.3f}{et:.3f}')

print(f'\nAll outputs: {os.path.abspath(OUTPUT_DIR)}')

COMPARISON TABLE
Method                             WT        TC        ET
-----------------------------------------------------------------
nnU-Net+TTA (clean)                0.890     0.901     0.819
nnU-Net+TTA (missing_T1CE)         0.888     0.797     0.595

v16b (clean)                       0.716     0.770     0.699
v16b (missing_T1CE)                0.661     0.433     0.270
v16b (missing_FLAIR)               0.583     0.718     0.596

All outputs: C:\Users\arnav\Desktop\MedBIND3D\MedBIND3D\medclipsam\MedCLIP-SAMv2\medbind3d_v16_outputs
